Install glycowork, Import all necessary helper functions from glycowork, and Import additional dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects
%matplotlib inline
import seaborn as sns
import networkx as nx
import os
!pip install --upgrade "glycowork @ git+https://github.com/BojarLab/glycowork.git@dev"
import glycowork
from glycowork.glycan_data.loader import glycomics_data_loader
from glycowork.network.biosynthesis import construct_network, plot_network, export_network, highlight_network, trace_diamonds, find_diamonds, choose_path
from glycowork.motif.draw import GlycoDraw
from scipy import stats
from scipy.stats import ttest_rel, ttest_ind
from statsmodels.stats.multitest import multipletests
from IPython.display import display
from collections import defaultdict


  Cloning https://github.com/BojarLab/glycowork.git (to revision dev) to /private/var/folders/lj/p78vp_151tb4lqfy3glspztm0000gn/T/pip-install-16mj1vwk/glycowork_97121039babc42dbb506326b0a1517ce
  Running command git clone --filter=blob:none --quiet https://github.com/BojarLab/glycowork.git /private/var/folders/lj/p78vp_151tb4lqfy3glspztm0000gn/T/pip-install-16mj1vwk/glycowork_97121039babc42dbb506326b0a1517ce
  Running command git checkout -b dev --track origin/dev
  Switched to a new branch 'dev'
  branch 'dev' set up to track 'origin/dev'.
  Resolved https://github.com/BojarLab/glycowork.git to commit c6cccb00e4750d28a9a10f86f25f988dc130947a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done


Extract daimond reactions from network

In [ ]:
#Function to extract reactions from diamond motifs in the network
def extract_diamond_reactions(net, diamonds):
    """
    Extract reactions from diamond motifs in the network.

    Parameters:
    net (networkx.Graph): The glycan biosynthesis network.
    diamonds (list of lists): List of diamond motifs, each containing nodes.

    Returns:
    list: A list of reactions associated with each diamond.
    """
    diamond_reactions = []
    for diamond in diamonds:
        reactions = []
        nodes = diamond

        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                if net.has_edge(nodes[i], nodes[j]):
                    reactions.append((nodes[i], nodes[j], net[nodes[i]][nodes[j]]['diffs'])) # Include start and end nodes with reaction
                elif net.has_edge(nodes[j], nodes[i]):
                    reactions.append((nodes[j], nodes[i], net[nodes[j]][nodes[i]]['diffs'])) # Include start and end nodes with reaction
        diamond_reactions.append(reactions)
    return diamond_reactions

Collect reaction preference data in the daimonds based on relative abundances

In [ ]:
# Function to prepare the reaction preferences data for all samples
def collect_reaction_preferences_data(diamond_reactions, net, sample_name, data_source):
    """
    Collect the preferred order of reaction based on intermediate abundance in a formatted structure.

    Parameters:
    diamond_reactions (list of tuples): Reactions extracted from diamonds.
    net (networkx.Graph): The glycan biosynthesis network containing node data.
    sample_name (str): Name of the sample being processed.
    data_source (str): Source of the data.

    Returns:
    List[dict]: List of dictionaries containing the table data.
    """
    # List to store each row of table data
    table_data = []
    source_name = data_source.split('.')[-1]

    for idx, diamond_react in enumerate(diamond_reactions):
        start_node_groups = defaultdict(list)

        # Group reactions by start node
        for start_node, end_node, diffs in diamond_react:
            abundance = net.nodes[end_node].get('abundance')
            start_node_groups[start_node].append((end_node, diffs, abundance))

        # Compare the abundance of intermediates and determine preference
        for start_node, reactions in start_node_groups.items():
            if len(reactions) == 2:
                intermediate1, diff1, abundance1 = reactions[0]
                intermediate2, diff2, abundance2 = reactions[1]

                # Ensure abundance values exist
                if abundance1 is not None and abundance2 is not None:
                    if abundance1 > abundance2:
                        preference = "forward"
                    elif abundance2 > abundance1:
                        preference = "reverse"
                    else:
                        preference = "no preference"
                    table_data.append({
                        'Diamond at': start_node,
                        'Glycan reaction addition 1': diff1,
                        'Abundance of the reaction 1': abundance1,
                        'Glycan reaction addition 2': diff2,
                        'Abundance of the reaction 2': abundance2,
                        'Reaction Preference': preference,
                        'Sample Name': sample_name,
                        'Data Source': source_name
                    })

    return table_data

Load the data, construct the network for each sample, display the table with reaction preferences

In [ ]:
# Load the data
df = glycomics_data_loader.human_skin_O_PMC5871710_BCC
# Main processing loop for all samples
all_data = []
data_source = 'glycomics_data_loader.human_skin_O_PMC5871710_BCC'
for sample_idx in range(1, df.shape[1]):
    sample_name = df.columns[sample_idx]

    # Construct the glycan network for the current sample
    abundances = df.iloc[:, sample_idx].values.tolist()
    net = construct_network(df.glycan.values.tolist(), abundances=abundances)

    # Find diamonds in the network
    diamonds = find_diamonds(net, mode="abundance")

    # Extract diamond reactions
    diamond_list = [list(d.values()) for d in diamonds]
    diamond_reactions = extract_diamond_reactions(net, diamond_list)

    # Collect reaction preferences data for each sample
    sample_data = collect_reaction_preferences_data(diamond_reactions, net, sample_name, data_source)
    all_data.extend(sample_data)
# Convert all collected data to a DataFrame
df_combined = pd.DataFrame(all_data)

# Sort the DataFrame by both 'Glycan reaction addition 1' and 'Glycan reaction addition 2'
df_combined = df_combined.sort_values(by=['Glycan reaction addition 1', 'Glycan reaction addition 2']).reset_index(drop=True)
df_combined = df_combined.sort_values(by=['Sample Name']).reset_index(drop=True)

Apply pair-wise Welch´s t-test, calculate error and display in table

In [ ]:
# List to store Welch's t-test results
results_welch = []

# Dictionary to track which pairs have been processed
processed_pairs = {}

# Group by 'Diff 1' and 'Diff 2' and apply Welch's t-test within each group
for (diff1, diff2), group in df_combined.groupby(['Glycan reaction addition 1', 'Glycan reaction addition 2']):
    # Always maintain consistent ordering of diff pairs
    diff1, diff2 = sorted([diff1, diff2])

    # Create a tuple for consistent tracking of processed pairs
    diff_pair = (diff1, diff2)

    # Check if this pair has already been processed
    if diff_pair in processed_pairs:
        # If already processed, add this group to the existing group
        processed_pairs[diff_pair] = pd.concat([processed_pairs[diff_pair], group])
    else:
        # Store the group for this new pair
        processed_pairs[diff_pair] = group

# Iterate through combined groups to perform the Welch's t-test
for diff_pair, combined_group in processed_pairs.items():
    if len(combined_group) >= 2:  # Ensure we have enough data points in each group
        # Perform Welch’s t-test on abundance of the intermediates
        group1 = combined_group['Abundance of the reaction 1'].dropna()
        group2 = combined_group['Abundance of the reaction 2'].dropna()

        if len(group1) > 1 and len(group2) > 1:  # Ensure adequate data in both groups
            t_stat, p_value = ttest_ind(group1, group2, equal_var=False)
            mean_diff = group1.mean() - group2.mean()
            std_dev = np.sqrt(group1.var() / len(group1) + group2.var() / len(group2))  # Combined std error

            # Assign p-value interpretation
            if p_value < 0.001:
                significance = '*** p < 0.001'
            elif p_value < 0.01:
                significance = '** p < 0.01'
            elif p_value < 0.05:
                significance = '* p < 0.05'
            else:
                significance = 'n.s., not significant'

            # Determine reaction preference based on mean difference between abundances
            if mean_diff > 0:
                reaction_preference = 'reverse'
            elif mean_diff < 0:
                reaction_preference = 'forward'
            else:
                reaction_preference = 'no preference'

            # Storing the results in the table
            results_welch.append({
                'Diff 1': diff_pair[0],
                'Diff 2': diff_pair[1],
                'Mean Difference (Abundance 1 - Abundance 2)': mean_diff,
                'Welch T-Statistic': t_stat,
                'Welch P-Value': p_value,
                'Significance': significance,
                'Reaction Preference': reaction_preference,
                'Error': std_dev  # Standard error of the mean difference
            })

# Convert results to DataFrame and display
results_welch_df = pd.DataFrame(results_welch)


Display the box plots showing forward or reverse reaction path preference

In [ ]:
# Define color map for reaction direction
color_map = {
    'forward': '#4682B4',        # Steel Blue (forward)
    'reverse': '#D2691E'         # Chocolate (reverse)
}

# Define SNFG colors for monosaccharides
snfg_color_map = {
    'Fuc': '#FF0000',            # Red (Fucose)
    'Gal': '#FFD700',            # Yellow (Galactose)
    'GlcNAc': '#87CEEB',         # Light Blue (N-Acetylglucosamine)
    'Neu5Ac': '#8A2BE2',         # Purple (N-Acetylneuraminic acid)
    'GalNAc': '#FFD700',         # yellow (N-Acetylgalactosamine)
    'Man': '#32CD32',            # Green (Mannose)
    'Glc': '#0000FF'             # Blue (Glucose)
}

# Define short labels for significance levels
short_significance_labels = {
    'n.s., not significant': 'n.s.',
    '* p < 0.05': '*',
    '** p < 0.01': '**',
    '*** p < 0.001': '***'
}
filtered_results_welch_df = results_welch_df[
    (results_welch_df['Significance'] != 'n.s., not significant')
]

# Sort data for clean presentation
results_welch_df_sorted =  filtered_results_welch_df.sort_values(
    by=['Diff 2', 'Mean Difference (Abundance 1 - Abundance 2)'],
    ascending=[False, False]
)

# Set up figure
plt.figure(figsize=(16, len(results_welch_df_sorted) * 0.8))
plt.rcParams['font.family'] = 'DejaVu Sans'

# Collect data for box plot and overlay distribution
box_plot_data = []
scatter_data_x = []
scatter_data_y = []
labels = []
colors = []
for i, (index, row) in enumerate(results_welch_df_sorted.iterrows()):
    diff1 = row['Diff 1']
    diff2 = row['Diff 2']
    addition_label = f"{diff1}, {diff2}"
    color = color_map.get(row['Reaction Preference'], color_map['forward'])

    # Add data for box plot from the original df_combined DataFrame
    combined_group = df_combined[(
        (df_combined['Glycan reaction addition 1'] == diff1) & (df_combined['Glycan reaction addition 2'] == diff2)) |
        ((df_combined['Glycan reaction addition 1'] == diff2) & (df_combined['Glycan reaction addition 2'] == diff1))
    ]
    abundance_data = combined_group['Abundance of the reaction 1'].values - combined_group['Abundance of the reaction 2'].values
    abundance_data = abundance_data[~np.isnan(abundance_data)]
    box_plot_data.append(abundance_data)
    labels.append(addition_label)
    colors.append(color)

    # Collect data points for overlay
    scatter_data_x.extend(abundance_data)
    scatter_data_y.extend([i + 1] * len(abundance_data))

# Create box plot for combined abundance data
box = plt.boxplot(box_plot_data, positions=np.arange(1, len(box_plot_data) + 1), vert=False, patch_artist=True, widths=0.6)

# Customize the boxes
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)

# Customize the whiskers, caps, and medians
for whisker in box['whiskers']:
    whisker.set(color='black', linewidth=1.5)
for cap in box['caps']:
    cap.set(color='black', linewidth=1.5)
for median in box['medians']:
    median.set(color='black', linewidth=1.5)

# Overlay data distribution as scatter points
plt.scatter(scatter_data_x, scatter_data_y, color='black', alpha=0.7, s=20, zorder=3)

# Add significance indicators above each box plot
for i, (index, row) in enumerate(results_welch_df_sorted.iterrows()):
    significance_label = short_significance_labels.get(row['Significance'], '')
    if significance_label:
        plt.text(np.max(box_plot_data[i]) + 3, i + 1.1, significance_label, ha='left', va='center', fontsize=10)

# Customize plot appearance
plt.xlabel('Mean Difference in Abundances of the Intermediates', fontsize=12)
plt.title('Forward vs Reverse Path Preference in Diamond Motif - Human Skin Dataset', fontsize=14)
plt.xticks(fontsize=10)
plt.gca().invert_yaxis()

# Customize y-tick labels without SNFG color coding
plt.yticks(np.arange(1, len(labels) + 1), labels, fontsize=10)

plt.tight_layout(rect=[0.02, 0, 1, 0.97])
plt.show()